# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

This feature vector aggregates March 2026 (month=2026-03) daily performance up to
one row per (client_hash_id, content_hash_id). It includes engineered ratios,
one categorical field (client's access_profile, one-hot encoded), and explicit
fills for missing values — documented below.

In [3]:
import pandas as pd
# Base aggregation from fact_content_daily_performance
base = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions_total,
        SUM(ga4_pageviews) AS ga4_pageviews_total,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_total,
        COUNT(*) AS days_observed,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS days_gsc_available
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

# Pull categorical context from dim_clients (access_profile)
clients = con.sql(f"""
    SELECT client_hash_id, access_profile
    FROM read_parquet('{BASE}/dim_clients.parquet')
""").df()

df = base.merge(clients, on="client_hash_id", how="left")

# --- Engineered features (ratios) ---
df["ctr"] = (df["gsc_clicks_total"] / df["gsc_impressions_total"]).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions_total"] / df["ga4_sessions_total"]).fillna(0)
df["gsc_coverage"] = (df["days_gsc_available"] / df["days_observed"]).fillna(0)

# --- Missing value fills (explicit, documented) ---
numeric_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total"]
df[numeric_cols] = df[numeric_cols].fillna(0)
df["access_profile"] = df["access_profile"].fillna("unknown")

# --- Categorical handling: one-hot encode access_profile ---
df = pd.get_dummies(df, columns=["access_profile"], prefix="access")

print("Feature vector shape:", df.shape)
df.head()

Feature vector shape: (331437, 17)


,client_hash_id,content_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_avg_position,ga4_sessions_total,ga4_pageviews_total,ga4_engaged_sessions_total,days_observed,days_gsc_available,ctr,engagement_rate,gsc_coverage,access_gsc_and_ga4,access_gsc_only,access_no_search_or_analytics_access,access_source_only_missing_client_dimension
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1.0,0.0,31,31.0,0.001073,0.0,1.000000,True,False,False,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,0.0,31,31.0,0.000000,0.0,1.000000,True,False,False,False
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,6.0,0.0,31,31.0,0.001066,0.0,1.000000,True,False,False,False
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,2.0,0.0,31,31.0,0.002629,0.0,1.000000,True,False,False,False
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,8.0,0.0,31,21.0,0.000000,0.0,0.677419,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Available before prediction? |
|---|---|---|---|
| `gsc_impressions_total` | Trailing GSC impressions summed over March | Filled with 0 (no GSC data = zero visibility) | Yes — purely past search data |
| `gsc_clicks_total` | Trailing GSC clicks summed over March | Filled with 0 | Yes — past-only |
| `gsc_avg_position` | Average search ranking position over the month | Filled with 0 (only for rows with no GSC data at all — flagged separately via `gsc_coverage`) | Yes — observed ranking, past |
| `ga4_sessions_total` | Trailing GA4 sessions summed over March | Filled with 0 (client may lack GA4 access) | Yes — past analytics data |
| `ga4_pageviews_total` | Trailing GA4 pageviews | Filled with 0 | Yes — past-only |
| `ga4_engaged_sessions_total` | Sessions with meaningful engagement | Filled with 0 | Yes — past-only |
| `days_observed` | Number of days this content appeared in March | Never missing (count) | Yes — historical fact |
| `days_gsc_available` | Days with usable GSC data | Never missing (count) | Yes — historical fact |
| `ctr` (engineered) | clicks/impressions ratio | 0 if impressions = 0 (avoids divide-by-zero) | Yes — derived from past-only inputs |
| `engagement_rate` (engineered) | engaged_sessions/sessions ratio | 0 if sessions = 0 | Yes — derived from past-only inputs |
| `gsc_coverage` (engineered) | days_gsc_available/days_observed | 0 if days_observed = 0 | Yes — derived from past-only inputs |
| `access_*` (categorical, one-hot) | Client's data-source access profile (gsc_and_ga4, gsc_only, etc.) | Filled with "unknown" before encoding | Yes — a client-level property known upfront, not tied to the outcome |

In [4]:
# Confirm no missing values remain after fills
missing_check = df.isnull().sum()
missing_check[missing_check > 0]


,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage hunt: I test my feature vector against three attack vectors — (1) a
label-derived column smuggled in as a feature, (2) a feature that overlaps the
future/outcome window, and (3) a suspiciously perfect single-feature predictor
(a signal that a feature is secretly encoding the label).

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build the same is_declining label as ML-04 (second half vs first half of March)
label_data = con.sql(f"""
    WITH halves AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id,
           (second_half < first_half) AS is_declining,
           (second_half - first_half) AS impression_change
    FROM halves
""").df()

attack_df = df.merge(label_data, on=["client_hash_id", "content_hash_id"])

feature_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total",
                 "days_observed", "days_gsc_available", "ctr", "engagement_rate", "gsc_coverage",
                 "access_gsc_and_ga4", "access_gsc_only",
                 "access_no_search_or_analytics_access", "access_source_only_missing_client_dimension"]

# --- Test A: honest baseline (my real feature vector) ---
X_train, X_test, y_train, y_test = train_test_split(
    attack_df[feature_cols], attack_df["is_declining"], test_size=0.3, random_state=0
)
honest_auc = roc_auc_score(y_test, LogisticRegression(max_iter=1000).fit(X_train, y_train).predict_proba(X_test)[:, 1])
print("Test A — Honest feature vector AUC:", round(honest_auc, 3))

# --- Test B: smuggle in a label-derived column (attack 1) ---
attack_cols_B = feature_cols + ["impression_change"]
X_train, X_test, y_train, y_test = train_test_split(
    attack_df[attack_cols_B], attack_df["is_declining"], test_size=0.3, random_state=0
)
leaked_auc_B = roc_auc_score(y_test, LogisticRegression(max_iter=1000).fit(X_train, y_train).predict_proba(X_test)[:, 1])
print("Test B — With label-derived column (impression_change) AUC:", round(leaked_auc_B, 3))

# --- Test C: single-feature perfect-predictor scan (attack 3) ---
print("\nTest C — Single-feature AUC scan (flag anything suspiciously close to 1.0):")
for col in feature_cols:
    try:
        single_auc = roc_auc_score(attack_df["is_declining"], attack_df[[col]].fillna(0))
        flag = "  <-- SUSPICIOUS" if single_auc > 0.95 or single_auc < 0.05 else ""
        print(f"  {col}: {single_auc:.3f}{flag}")
    except Exception as e:
        print(f"  {col}: could not test ({e})")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Test A — Honest feature vector AUC: 0.804
Test B — With label-derived column (impression_change) AUC: 1.0

Test C — Single-feature AUC scan (flag anything suspiciously close to 1.0):
  gsc_impressions_total: 0.792
  gsc_clicks_total: 0.607
  gsc_avg_position: 0.789
  ga4_sessions_total: 0.583
  ga4_pageviews_total: 0.584
  ga4_engaged_sessions_total: 0.517
  days_observed: 0.558
  days_gsc_available: 0.814
  ctr: 0.606
  engagement_rate: 0.517
  gsc_coverage: 0.811
  access_gsc_and_ga4: 0.468
  access_gsc_only: 0.533
  access_no_search_or_analytics_access: 0.500
  access_source_only_missing_client_dimension: 0.500


**Findings:**
- Honest AUC (all 15 real features) = 0.804 — a reasonable, non-suspicious score.
- Smuggling in `impression_change` (the label-derived quantity) pushed AUC to 1.0 —
  confirming this column must never be used as a feature.
- Single-feature scan: no individual feature exceeded 0.95 AUC on its own. The
  highest were `days_gsc_available` (0.814) and `gsc_coverage` (0.811) — both
  plausible (content with more consistent GSC coverage naturally has more stable
  performance signal), not label leakage.
- Conclusion: the 15-feature vector passes the leakage hunt. `impression_change`
  is excluded going forward.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

***Fields deliberately excluded from the feature vector, with reasons:***

- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`,
  `ai_other` — present (non-null) in ~69% of rows, but non-zero traffic in only
  3,177 of 9,841,378 rows (~0.03%). Near-universally zero, so these columns carry
  almost no usable signal for this month's slice; excluded rather than adding
  noise to the feature vector.

In [7]:
sparsity_check2 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ai_chatgpt > 0 THEN 1 ELSE 0 END) AS ai_chatgpt_nonzero
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
sparsity_check2

,total_rows,ai_chatgpt_nonzero
0,9841378,3177.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.